<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing
**Pipeline:** ESG PDF reports + NewsAPI articles → cleaned, chunked, combined CSV

**Outputs pushed to `data/processed/` on GitHub:**
- `esg_chunks.csv` — chunked text from 12 company ESG PDFs
- `news_articles.csv` — news articles fetched from NewsAPI
- `combined.csv` — merged dataset, input for filtering/scoring notebook

> **Only needs to be re-run when source data changes.**  
> The filtering/scoring notebook loads directly from GitHub — no reprocessing needed.

In [ ]:
# --- SETUP ---
# Install deps, download shared ESG PDFs, set up output dirs
# GITHUB_TOKEN must be set in Colab Secrets 
# Generate one at: github.com/settings/tokens (scope: repo)

!pip install pdfplumber newsapi-python gdown -q

import os
from google.colab import userdata

FOLDER_ID  = '1SUewYSR3PYYAMUHi9Egqc3Bd8wA0zKeY'
DATA_DIR   = '/content/data'
OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
GIT_EMAIL  = 'aatman.dwivedi@student.uts.edu.au'
GIT_NAME   = 'Swas26'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download PDFs from shared Drive folder (no sign-in needed)
if not os.path.exists(DATA_DIR):
    print("Downloading shared data folder...")
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O {DATA_DIR}
else:
    print("Data already present, skipping download.")

Data already present, skipping download.


In [15]:
# --- PART 1: ESG PDF CHUNKING ---
# Reads each company PDF, extracts raw text, splits into ~400-word chunks
# Chunking is necessary because ClimateBERT has a max token limit per input

import pdfplumber
import pandas as pd

filename_to_company = {
    'Woolworths Australia.pdf': 'Woolworths Australia',
    'UniQlo.pdf': 'UniQlo',
    'Santos.pdf': 'Santos',
    'Qantas.pdf': 'Qantas',
    'NBN.pdf': 'NBN',
    'Muji.pdf': 'Muji',
    'LVMH.pdf': 'LVMH',
    'Gucci.pdf': 'Gucci',
    'Coles Group.pdf': 'Coles Group',
    'CBA.pdf': 'CBA',
    'ANZ.pdf': 'ANZ',
    'AGL.pdf': 'AGL',
}

all_chunks = []
CHUNK_SIZE = 400

for filename, company_name in filename_to_company.items():
    filepath = os.path.join(DATA_DIR, filename)

    if not os.path.exists(filepath):
        print(f"MISSING: {filename} — skipping")
        continue

    full_text = ''
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + ' '

    words = full_text.split()
    chunks_added = 0

    for i in range(0, len(words), CHUNK_SIZE):
        chunk = ' '.join(words[i:i+CHUNK_SIZE])
        if len(chunk.strip()) > 50:  # skip near-empty chunks
            all_chunks.append({
                'company': company_name,
                'source_type': 'esg_report',
                'text': chunk
            })
            chunks_added += 1

    print(f"{company_name}: {len(words)} words → {chunks_added} chunks")

esg_df = pd.DataFrame(all_chunks)
esg_df.to_csv(f'{OUTPUT_DIR}/esg_chunks.csv', index=False)
print(f"\nDone — {len(esg_df)} total ESG chunks saved")

MISSING: Woolworths Australia.pdf — skipping
MISSING: UniQlo.pdf — skipping
Santos: 141190 words → 353 chunks
MISSING: Qantas.pdf — skipping
NBN: 11146 words → 28 chunks
Muji: 58365 words → 146 chunks
LVMH: 53393 words → 134 chunks
Gucci: 6814 words → 18 chunks
Coles Group: 40158 words → 101 chunks
CBA: 4104 words → 11 chunks
MISSING: ANZ.pdf — skipping
AGL: 114962 words → 288 chunks

Done — 1079 total ESG chunks saved


In [16]:
# --- PART 2: NEWS ARTICLE SCRAPING ---
# Fetches sustainability-related news for each company via NewsAPI
# Free tier limit: 30 articles/company, 1 req/sec — don't re-run unnecessarily

from newsapi import NewsApiClient
import time

api = NewsApiClient(api_key='822e469b13934cf2adc43ce8872be994')

companies = [
    'AGL', 'CBA', 'ANZ', 'Qantas', 'Santos', 'NBN',
    'Muji', 'LVMH', 'UniQlo', 'Woolworths Australia', 'Coles Group', 'Gucci'
]

all_articles = []

for company in companies:
    response = api.get_everything(
        q=f'{company} sustainability OR environment OR ESG OR emissions',
        language='en',
        page_size=30,
        sort_by='relevancy'
    )

    for article in response['articles']:
        all_articles.append({
            'company': company,
            'source_type': 'news',
            'text': str(article['title']) + '. ' + str(article['description']),
            'published_at': article['publishedAt'],
            'source': article['source']['name']
        })

    print(f"{company}: {len(response['articles'])} articles fetched")
    time.sleep(1)

news_df = pd.DataFrame(all_articles)
news_df.dropna(subset=['text'], inplace=True)
news_df.to_csv(f'{OUTPUT_DIR}/news_articles.csv', index=False)
print(f"\nDone — {len(news_df)} articles saved")

AGL: 20 articles fetched
CBA: 29 articles fetched
ANZ: 29 articles fetched
Qantas: 28 articles fetched
Santos: 30 articles fetched
NBN: 30 articles fetched
Muji: 13 articles fetched
LVMH: 30 articles fetched
UniQlo: 28 articles fetched
Woolworths Australia: 30 articles fetched
Coles Group: 27 articles fetched
Gucci: 30 articles fetched

Done — 324 articles saved


In [17]:
# --- PART 3: COMBINE ESG + NEWS ---
# Merges both sources into one dataset
# combined.csv is the single input for the filtering/scoring notebook

esg_df  = pd.read_csv(f'{OUTPUT_DIR}/esg_chunks.csv')
news_df = pd.read_csv(f'{OUTPUT_DIR}/news_articles.csv')

news_df = news_df[['company', 'source_type', 'text']]  # align columns

combined_df = pd.concat([esg_df, news_df], ignore_index=True)
combined_df.to_csv(f'{OUTPUT_DIR}/combined.csv', index=False)

print(f"ESG chunks:     {len(esg_df)}")
print(f"News articles:  {len(news_df)}")
print(f"Total combined: {len(combined_df)}")
print(combined_df['source_type'].value_counts())

ESG chunks:     1079
News articles:  324
Total combined: 1403
source_type
esg_report    1079
news           324
Name: count, dtype: int64


In [18]:
# --- PUSH OUTPUTS TO GITHUB ---
# Copies processed CSVs into the repo and pushes to data/processed/
# Teammates just run `git pull` to get the latest files — no reprocessing needed

import shutil

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/repo'

# Clone repo
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q

# Copy all CSVs from outputs into repo
processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f"Copied {f}")

# Commit and push
!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update preprocessed data outputs"
!cd {REPO_DIR} && git push

print("\nAll outputs pushed to GitHub → data/processed/")

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.